<a href="https://colab.research.google.com/github/Neeti1987/Pathway-GEM-scripts/blob/main/Phenylalanine_pathway_specific_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === STEP 1: Install dependencies ===
!pip install cobra pandas

import cobra
from cobra import Model, Reaction, Metabolite
import pandas as pd
from google.colab import files

# === STEP 2: Upload KO annotation file ===
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# === STEP 3: Parse KO annotations genome-wise ===
genome_kos = {}
with open(filename, "r") as f:
    for line in f:
        if not line.strip(): continue
        parts = line.strip().split("\t")
        if len(parts) < 2: continue
        genome_id, ko = parts[0], parts[1]
        genome_prefix = genome_id.split("_")[0]
        genome_kos.setdefault(genome_prefix, set()).add(ko)

# === STEP 4: KO logic for M00024 (Phenylalanine biosynthesis) ===
def check_phe_module(ko_list):
    status = {}
    status["Chorismate_mutase"] = any(k in ko_list for k in ["K01850","K04092","K14187","K04093","K04516","K06208","K06209"])
    status["Prephenate_dehydratase"] = any(k in ko_list for k in ["K01713","K04518","K05359","K14170"])
    status["Aromatic_transaminase"] = any(k in ko_list for k in ["K00832","K00838"])
    complete = all(status.values())
    return complete, status

# === STEP 5: Build phenylalanine pathway model ===
def build_phe_model():
    model = Model("Phenylalanine_pathway")

    # Metabolites (KEGG C numbers)
    chorismate = Metabolite("C00251_c", compartment="c")
    prephenate = Metabolite("C00254_c", compartment="c")
    phenylpyruvate = Metabolite("C00166_c", compartment="c")
    phe = Metabolite("C00079_c", compartment="c")        # Phenylalanine
    glu = Metabolite("C00025_c", compartment="c")        # Glutamate
    akg = Metabolite("C00026_c", compartment="c")        # 2-Oxoglutarate
    h2o = Metabolite("C00001_c", compartment="c")
    co2 = Metabolite("C00011_c", compartment="c")

    # Reactions
    r01715 = Reaction("R01715_Chorismate_mutase")
    r01715.add_metabolites({chorismate:-1, prephenate:1})

    r01373 = Reaction("R01373_Prephenate_dehydratase")
    r01373.add_metabolites({prephenate:-1, phenylpyruvate:1, h2o:1, co2:1})

    r00694 = Reaction("R00694_Aromatic_transaminase")
    r00694.add_metabolites({phenylpyruvate:-1, glu:-1, phe:1, akg:1})

    model.add_reactions([r01715, r01373, r00694])

    # Exchanges for all metabolites (permissive style)
    for met in [chorismate, prephenate, phenylpyruvate, phe, glu, akg, h2o, co2]:
        ex = Reaction(f"EX_{met.id}")
        ex.add_metabolites({met:1})
        ex.lower_bound = -1000
        ex.upper_bound = 1000
        model.add_reactions([ex])

    # Demand reaction for phenylalanine
    dm_phe = Reaction("DM_C00079")
    dm_phe.add_metabolites({phe:-1})
    dm_phe.lower_bound = 0
    dm_phe.upper_bound = 1000
    model.add_reactions([dm_phe])
    model.objective = dm_phe

    return model

# === STEP 6: Run genome-wise analysis ===
summary = []
for genome, kos in genome_kos.items():
    complete, status = check_phe_module(kos)
    flux_value = None
    if complete:
        model = build_phe_model()
        sol = model.optimize()
        flux_value = sol.objective_value
    summary.append({
        "Genome": genome,
        **status,
        "Module_complete": complete,
        "Flux_to_Phenylalanine": flux_value
    })

# === STEP 7: Output results ===
df = pd.DataFrame(summary)
print(df)
df.to_csv("phenylalanine_flux_genomewise.tsv", sep="\t", index=False)

# === STEP 8: Download TSV ===
files.download("phenylalanine_flux_genomewise.tsv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.8/141.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.7 MB/s eta 0:00:00


Saving PORTIERAKO.tsv to PORTIERAKO.tsv
    Genome  Chorismate_mutase  Prephenate_dehydratase  Aromatic_transaminase  \
0    ADCAI              False                    True                  False   
1    AFCAI              False                    True                  False   
2      BTB              False                    True                  False   
3      BTQ              False                    True                  False   
4   BTQVLC              False                    True                  False   
5     BTZ1              False                    True                  False   
6    China              False                    True                  False   
7   China1              False                    True                  False   
8    MEAM1              False                    True                  False   
9     PeMo              False                    True                  False   
10    SiSi              False                    True                  False   


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# === STEP 6b: Simulate metabolic complementation ===
def complement_phe_model(missing_step="chorismate_mutase"):
    model = build_phe_model()

    # By default, uptake is blocked. We selectively allow host-derived metabolite uptake.
    if missing_step == "chorismate_mutase":
        # Allow uptake of prephenate (host supplies it)
        model.reactions.get_by_id("EX_C00254_c").lower_bound = -10
    elif missing_step == "aromatic_transaminase":
        # Allow uptake of phenylpyruvate (host supplies it)
        model.reactions.get_by_id("EX_C00166_c").lower_bound = -10

    # Re-run optimization
    sol = model.optimize()
    return sol.objective_value

# Example: test complementation for one genome
flux_with_complement = complement_phe_model("chorismate_mutase")
print("Flux to phenylalanine with host complementation:", flux_with_complement)


Flux to phenylalanine with host complementation: 1000.0


In [ ]:
# === STEP 6b: Complementation test function ===
def complement_phe_model(missing_step="chorismate_mutase"):
    model = build_phe_model()

    # Allow host-derived metabolite uptake depending on missing step
    if missing_step == "chorismate_mutase":
        model.reactions.get_by_id("EX_C00254_c").lower_bound = -10   # prephenate uptake
    elif missing_step == "aromatic_transaminase":
        model.reactions.get_by_id("EX_C00166_c").lower_bound = -10   # phenylpyruvate uptake

    sol = model.optimize()
    return sol.objective_value

# === STEP 6c: Genome-wise analysis with complementation ===
summary = []
for genome, kos in genome_kos.items():
    complete, status = check_phe_module(kos)

    # Default flux (no complementation)
    flux_value = None
    if complete:
        model = build_phe_model()
        sol = model.optimize()
        flux_value = sol.objective_value

    # Complementation flux (simulate host provision)
    flux_complement = None
    if not complete:
        # If chorismate mutase missing, test prephenate uptake
        if not status["Chorismate_mutase"]:
            flux_complement = complement_phe_model("chorismate_mutase")
        # If aromatic transaminase missing, test phenylpyruvate uptake
        elif not status["Aromatic_transaminase"]:
            flux_complement = complement_phe_model("aromatic_transaminase")

    summary.append({
        "Genome": genome,
        **status,
        "Module_complete": complete,
        "Flux_to_Phenylalanine": flux_value,
        "Flux_with_Complementation": flux_complement
    })

# === STEP 7: Output results ===
df = pd.DataFrame(summary)
print(df)
df.to_csv("phenylalanine_flux_complementation.tsv", sep="\t", index=False)
files.download("phenylalanine_flux_complementation.tsv")


    Genome  Chorismate_mutase  Prephenate_dehydratase  Aromatic_transaminase  \
0    ADCAI              False                    True                  False   
1    AFCAI              False                    True                  False   
2      BTB              False                    True                  False   
3      BTQ              False                    True                  False   
4   BTQVLC              False                    True                  False   
5     BTZ1              False                    True                  False   
6    China              False                    True                  False   
7   China1              False                    True                  False   
8    MEAM1              False                    True                  False   
9     PeMo              False                    True                  False   
10    SiSi              False                    True                  False   
11      TV              False           

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>